In [ ]:
import os
from pprint import pprint

import pandas as pd
import numpy as np
import pyreadstat
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

In [ ]:
# --- Shared color palette (module 3.3) ---
PALETTE = ["#7BB3B2","#65A6BD","#C997AF","#B8B0D3","#F4CF97","#98B9A0","#F6DECD"]

# --- Plotly template: set once, every chart inherits it (module 3.10, Principle 5) ---
nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    colorway=PALETTE,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40)
)
pio.templates['nso'] = nso_template
pio.templates.default = 'nso'

In [ ]:
# --- Plotly toolbar config ---
PLOTLY_CONFIG = {
    'displaylogo': False,
    'modeBarButtonsToRemove': ['select2d', 'lasso2d', 'autoScale2d'],
    'toImageButtonOptions': {'format': 'png', 'width': 1200, 'height': 700, 'scale': 2}
}

In [ ]:
# %% [Load data]
df, metadata = pyreadstat.read_sav('../../data/raw_0/ZM_LFS_DATASET2024_Annual_10percent.sav')

In [ ]:
# %% [Inspect column labels]
print(set(metadata.column_labels))

In [ ]:
# %% [Select columns of interest]
# This cell was written after investigating the above result and picking
# column names that were likely to contain interesting data

cols = [
    # --- Demographics & Background ---
    "Is ... Male or Female?",
    "How old was ... at (his/her) last birthday?",
    "What is the highest grade/level of education that ... has successfully completed?",
    "What is ...'s current marital status?",
    "What is ...'s relationship to the head of the household?",
    "1. Province",
    "2. District",

    # --- Employment & Work ---
    "In the main job/business that (NAME) has, is she/he...",
    "INDUSTRY",
    "Occupation",
    "How many hours does (NAME) usually work per week in his/her...? Main job",
    "How many hours does (NAME) usually work per week in his/her...? OVERALL TOTAL",
    "What is the frequency of .....'s income/earnings in his/her main job?",
    "Would (NAME) want to work more hours per week than usually worked, provided the extra hours are paid?",
    "Is ?. employed on the basis of a written contract or an oral agreement?",

    # --- Income & Earnings ---
    "What is your annually/monthly/weekly/daily/hourly wage or salary before deductions?",
    "What are your annual/monthly/weekly/daily/hourly earnings after expenses?",
    "At what age did NAME start work for the first time in his /her life",

    # --- Time Use: Household Activities ---
    "During the last 7 days how much time did  (NAME) spend on Cleaning the house, washing clothes, cooking or shopping for the household",
    "During the last 7 days how much time did  (NAME) spend on Fetching water from natural or public sources for use by the household",
    "During the last 7 days how much time did (NAME) spend on Collecting firewood or other natural products for use as fuel by the household",
    "In the last 7 days, how much time did (NAME) spend on Leisure e.g., playing sports, watching TV etc.?",
    "In the last 7 days, how much time did (NAME) spend on Personal care e.g bathing, eating and sleeping?",
    "In the last 7 days, how much time did name spend travelling from home to\xa0place\xa0of\xa0work",

    # --- Time Use: Hours by Day of Week (Main Job) ---
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Monday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Tuesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Wednesday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Thursday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Friday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Saturday Main job?",
    "Thinking about the last 7 Days, how many hours  ?? work his/her on Sunday Main job?",

    # --- Financial Inclusion ---
    "P.20. Do you own a mobile phone",
    "P.21. Do you have a mobile money account in your own name",
    "P.23. How often do you use mobile money?",
    "P.25.  On a scale of 1 to 4, Do you find mobile money services to be cheap or expensive?",
    "P.30.A Savings at a bank",
    "P.30.G.Savings with savings group",
    "P.30.D.Savings that you keep on your mobile phone",
    "What method do you mainly use to pay for food/groceries?",

    # --- Education ---
    "Can... read and write in any language?",
    "Has... ever attended school?",
    "Is (NAME) currently attending school?",
    "Have (NAME) ever repeated any level of schooling any point in time?",
    "At what age did (NAME) begin school?",
]

# For each label we pick the original column name as it appears in df.
names_to_labels = metadata.column_names_to_labels
names_to_labels_reduced = {}
names = []
for col in cols:
    for name, label in names_to_labels.items():
        if label != col:
            continue
        names.append(name)
        names_to_labels_reduced[name] = label
pprint(names_to_labels_reduced)

In [ ]:
# %% [Filter dataframe to columns of interest]
df = df[names]

In [ ]:
df.groupby("A3")["A2"].mean().values

In [ ]:
# %% [Extract value labels from metadata]
variable_value_labels = metadata.variable_value_labels

In [ ]:
# %% [Chart 1: Age Distribution — Histogram]
# Pattern: Distribution (module 3.4, Pattern 1)
# Structure: single Histogram trace + layout (3.10 Principle 1)
# Reference line marks the median (module 3.6)

# ----- Step 1: Prepare -----
age = df["A3"].dropna()
median_age = age.median()

# ----- Step 2: Plot -----
fig = go.Figure(
    go.Histogram(x=age, nbinsx=20, marker_color=PALETTE[0], name='Age')
)

fig.add_vline(
    x=median_age, line_dash='dash', line_color='red',
    annotation_text=f'Median: {median_age:.0f}',
    annotation_position='top right'
)

fig.update_layout(
    title='Age Distribution',
    xaxis_title='Age',
    yaxis_title='Count'
)
fig.show(config=PLOTLY_CONFIG)

# %% [Inspect gender value labels]
variable_value_labels["A2"]

In [ ]:
# %% [Chart 2: Age Pyramid — Two Horizontal Bar Traces]
# A population pyramid is two Bar traces going in opposite directions
# (3.10 Principle 2: multiple things on one chart = multiple traces)

# ----- Step 1: Prepare -----
bins = list(range(0, 85, 5))
age_labels = [f"{b} - {b+4}" for b in bins[:-1]]

df_pyr = df[["A2", "A3"]].dropna().copy()
df_pyr["age_group"] = pd.cut(df_pyr["A3"], bins=bins, labels=age_labels, right=False)
df_pyr["A2"] = df_pyr["A2"].map(variable_value_labels["A2"])

male = df_pyr[df_pyr["A2"] == 'Male'].groupby("age_group").size()
female = df_pyr[df_pyr["A2"] == 'Female'].groupby("age_group").size()

# ----- Step 2: Plot -----
fig = go.Figure()

fig.add_trace(go.Bar(
    y=age_labels, x=-male.reindex(age_labels, fill_value=0).values,
    name='Male', orientation='h', marker_color=PALETTE[0]
))
fig.add_trace(go.Bar(
    y=age_labels, x=female.reindex(age_labels, fill_value=0).values,
    name='Female', orientation='h', marker_color=PALETTE[2]
))

fig.update_layout(
    title='Age Pyramid',
    xaxis_title='Population Count',
    barmode='overlay',
    bargap=0.1,
    height=600
)
fig.show(config=PLOTLY_CONFIG)